# CIFAR-DenseNet121 block-boundary experiment — seed 1

This is a from-scratch, self-contained comparison using CIFAR-100 and Kaggle
T4 x2. It needs only the CIFAR-100 input and the `github_token` secret.

Phase 1 trains a randomly initialized CIFAR-DenseNet121 with the
fixed base recipe: SGD, LR 0.1, milestones 100/150, gamma 0.1, momentum 0.9,
weight decay 5e-4, and 200 recipe epochs. Starting at epoch 200, every strict
raw validation best is saved. Theta_P is accepted only after **150 consecutive
epochs without another raw validation best**. Those exact 150 epochs are the
Vanilla comparison trajectory; Vanilla is not restarted.

Phase 2 loads the byte-identical theta_P into two fresh processes:

- GPU0: recurrent E-driven O, 150 SGD epochs;
- GPU1: recurrent O-only, 150 SGD epochs.

Both projected arms use raw validation-best rollback with patience 10. There is
no Bypass arm in this experiment. E-driven O scans all 4 DenseBlock boundaries,
lets structural E-gain select WHERE, and projects only the winner. The official
CIFAR-100 test set is never constructed.


In [ ]:
import hashlib, json, os, shutil, subprocess, sys, threading, time, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient

MAIN_URL = "https://github.com/duyh80456-code/Counterfactual-Projection.git"
REPO = Path("/kaggle/working/counterfactual-projection")
REFERENCE_URL = "https://github.com/duyh80456-code/One-Shot-TAS-CCIL.git"
REFERENCE = Path("/kaggle/working/One-Shot-TAS-CCIL")
GROMO_URL = "https://github.com/growingnet/gromo.git"
GROMO_COMMIT = "8d19107b61a9459a9021065a329b699adcb0f25b"
GROMO = Path("/kaggle/working/gromo")
RUN_ROOT = Path("/kaggle/working/densenet121_seed1_stall150_v1")
OUTPUT = RUN_ROOT / "phase2"

def private_clone(url, destination, branch):
    token = UserSecretsClient().get_secret("github_token").strip()
    if not token: raise RuntimeError("Kaggle Secret github_token is unavailable")
    askpass = Path("/kaggle/working/.counterfactual_git_askpass.py")
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update(GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass),
               GIT_TERMINAL_PROMPT="0")
    try:
        subprocess.run(["git", "clone", "--branch", branch, "--single-branch",
                        url, str(destination)], env=env, check=True)
    finally:
        askpass.unlink(missing_ok=True)

for checkout in (REPO, REFERENCE, GROMO):
    if checkout.exists(): shutil.rmtree(checkout)
OUTPUT.mkdir(parents=True, exist_ok=True)
private_clone(MAIN_URL, REPO, "main")
private_clone(REFERENCE_URL, REFERENCE, "ccil-residual-capacity")
subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                GROMO_URL, str(GROMO)], check=True)
subprocess.run(["git", "-C", str(GROMO), "fetch", "--depth", "1",
                "origin", GROMO_COMMIT], check=True)
subprocess.run(["git", "-C", str(GROMO), "checkout", "--detach",
                GROMO_COMMIT], check=True)
print("main:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(GROMO)], check=True)
GROMO_SRC = GROMO / "src"
RUNTIME_PYTHONPATH = os.pathsep.join(filter(None, (
    str(REPO), str(GROMO_SRC), str(REFERENCE), os.environ.get("PYTHONPATH", ""))))
test_env = os.environ.copy(); test_env.update(PYTHONPATH=RUNTIME_PYTHONPATH)
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO,
               env=test_env, check=True)
import torch
if torch.cuda.device_count() != 2:
    raise RuntimeError(f"Select T4 x2; found {torch.cuda.device_count()} GPU(s)")
required_cifar_files = {"train", "test", "meta"}
cifar_dirs = []
for directory, _, filenames in os.walk("/kaggle/input", followlinks=True):
    path = Path(directory)
    if (path.name == "cifar-100-python" and
            required_cifar_files.issubset(filenames)):
        cifar_dirs.append(path.parent)
cifar_dirs = sorted(set(cifar_dirs), key=str)
if not cifar_dirs:
    visible = []
    for directory, dirnames, filenames in os.walk(
            "/kaggle/input", followlinks=True):
        visible.append({
            "directory": directory,
            "directories": sorted(dirnames)[:10],
            "files": sorted(filenames)[:10],
        })
        if len(visible) == 20:
            break
    raise FileNotFoundError(
        "Attach CIFAR-100 containing cifar-100-python/{train,test,meta}. "
        f"Visible input inventory: {visible}")
DATA_ROOT = cifar_dirs[0]
print("CIFAR-100 root:", DATA_ROOT)


In [ ]:
import importlib.util

def load_checkout_module(name, relative_path):
    path = REPO / relative_path
    if not path.is_file():
        raise RuntimeError(
            f"Repository helper is missing: {path}. Run the clone/bootstrap "
            "cells above before this cell.")
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Cannot load repository helper: {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    try:
        spec.loader.exec_module(module)
    except Exception:
        sys.modules.pop(name, None)
        raise
    return module

checkpoint_discovery = load_checkout_module(
    "_counterfactual_kaggle_checkpoint_discovery",
    "experiments/kaggle_checkpoint_discovery.py")
plateau_protocol = load_checkout_module(
    "_counterfactual_plateau_protocol", "experiments/plateau_protocol.py")
discover_checkpoints = checkpoint_discovery.discover_checkpoints
BestCheckpointStallDetector = plateau_protocol.BestCheckpointStallDetector

SEED = 1
PHASE1_OUTPUT = RUN_ROOT / "vanilla_stall"
PHASE1_OUTPUT.mkdir(parents=True, exist_ok=True)
schedule_id = "cifar-densenet121-sgd-multistep-200-v1-post200-val-best"

def copy_checkpoint(source, destination):
    source, destination = Path(source), Path(destination)
    if source.resolve() != destination.resolve():
        shutil.copy2(source, destination)

def compatible_training_lineage(candidate, reference):
    # Compare state-producing invariants, not incidental report metadata.
    required_state = {
        "model", "optimizer", "scheduler", "rng",
        "train_loader_generator_state", "train_indices", "trigger_indices",
        "evaluation_indices", "source_tuning_indices", "history", "epoch"}
    if not required_state.issubset(candidate) or not required_state.issubset(reference):
        return False
    protocol_fields = (
        "seed", "dataset", "architecture", "input_size", "learning_rate",
        "weight_decay", "batch_size", "schedule_id", "optimizer",
        "model_state_lineage", "train_indices_sha256",
        "validation_indices_sha256", "tuning_indices_sha256")
    left, right = candidate.get("protocol", {}), reference.get("protocol", {})
    if any(left.get(key) != right.get(key) for key in protocol_fields):
        return False
    for key in ("train_indices", "trigger_indices", "evaluation_indices",
                "source_tuning_indices"):
        if list(candidate[key]) != list(reference[key]):
            return False
    scheduler_fields = ("kind", "milestones", "gamma", "recipe_epochs")
    if any(candidate["scheduler"].get(key) != reference["scheduler"].get(key)
           for key in scheduler_fields):
        return False
    optimizer_fields = ("momentum", "dampening", "weight_decay", "nesterov")
    candidate_groups = candidate["optimizer"].get("param_groups", [])
    reference_groups = reference["optimizer"].get("param_groups", [])
    if len(candidate_groups) != len(reference_groups):
        return False
    if any(any(a.get(key) != b.get(key) for key in optimizer_fields)
           for a, b in zip(candidate_groups, reference_groups)):
        return False
    candidate_shapes = {key: tuple(value.shape)
                        for key, value in candidate["model"].items()}
    reference_shapes = {key: tuple(value.shape)
                        for key, value in reference["model"].items()}
    return candidate_shapes == reference_shapes

def is_post200_raw_best_snapshot(item):
    payload = item["payload"]
    if not isinstance(payload, dict) or "history" not in payload:
        return False
    epoch = int(payload.get("epoch", -1))
    rows = [row for row in payload["history"]
            if 200 <= int(row["epoch"]) <= epoch]
    if not rows:
        return False
    observed = max(rows, key=lambda row: float(row["validation_accuracy"]))
    return int(observed["epoch"]) == epoch

checkpoint_kinds = {
    "unified_vanilla_progress", "vanilla_validation_best_checkpoint",
    "vanilla_significant_best_checkpoint", "vanilla_exact_best_diagnostic",
    "plateau_fork_checkpoint"}
input_discovered, input_rejected = discover_checkpoints(
    "/kaggle/input", RUN_ROOT / "input_discovery_cache",
    kind=checkpoint_kinds)
local_discovered, local_rejected = discover_checkpoints(
    PHASE1_OUTPUT, RUN_ROOT / "local_discovery_cache",
    kind=checkpoint_kinds)
discovered_by_identity = {}
for item in [*input_discovered, *local_discovered]:
    identity = (
        item["sha256"], item["payload"].get("kind"),
        int(item["payload"].get("epoch", -1)))
    discovered_by_identity.setdefault(identity, item)
discovered = list(discovered_by_identity.values())
rejected_checkpoints = [*input_rejected, *local_rejected]
print("Phase-1 checkpoint inventory:", [{
    "kind": item["payload"].get("kind"),
    "epoch": int(item["payload"].get("epoch", -1)),
    "schedule_id": item["payload"].get("protocol", {}).get("schedule_id"),
    "seed": item["payload"].get("protocol", {}).get("seed"),
    "source": item["source"],
} for item in discovered])
progresses = [item for item in discovered
              if item["payload"].get("kind") == "unified_vanilla_progress"]
bests = [item for item in discovered if is_post200_raw_best_snapshot(item)]
compatible = [item for item in progresses
              if item["payload"].get("protocol", {}).get("seed") == SEED
              and item["payload"].get("protocol", {}).get(
                  "schedule_id") == schedule_id
              and item["payload"].get("protocol", {}).get(
                  "dataset") == "CIFAR-100"
              and item["payload"].get("protocol", {}).get(
                  "architecture") == "CIFAR-DenseNet121"
              and item["payload"].get("protocol", {}).get(
                  "batch_size") == 64
              and item["payload"].get("protocol", {}).get(
                  "optimizer") ==
                  "SGD(momentum=0.9, weight_decay=0.0005)"
              and item["payload"].get("protocol", {}).get(
                  "model_state_lineage") == f"random_init_densenet121_seed_{SEED}"]
PHASE1_MAX_EPOCH = 800
if not compatible:
    # Older unified runs used the same base recipe. Choose the closest full
    # checkpoint at or before the observed post-200 validation best and replay
    # forward; never synthesize weights from history alone.
    legacy = [item for item in progresses
              if item["payload"].get("protocol", {}).get("seed") == SEED
              and item["payload"].get("protocol", {}).get(
                  "schedule_id") in {
                  "cifar-resnet18-sgd-multistep-200-v3-significant-fork",
                  "cifar-resnet18-sgd-multistep-200-v4-validation-best"}]
    legacy = []
    if legacy:
        old_progress = max(
            legacy, key=lambda item: int(item["payload"]["epoch"]))
        old_payload = old_progress["payload"]
        old_history = old_payload["history"]
        post200 = [row for row in old_history if int(row["epoch"]) >= 200]
        if not post200:
            raise RuntimeError("Legacy progress does not reach epoch 200")
        old_best_row = max(
            post200, key=lambda row: row["validation_accuracy"])
        old_best_epoch = int(old_best_row["epoch"])
        replay_sources = [item for item in discovered
                          if item["payload"].get("protocol", {}).get(
                              "seed") == SEED
                          and item["payload"].get("protocol", {}).get(
                              "schedule_id") ==
                              old_payload["protocol"].get("schedule_id")
                          and int(item["payload"].get("epoch", -1)) <=
                              old_best_epoch]
        if replay_sources:
            replay_source = max(
                replay_sources,
                key=lambda item: int(item["payload"]["epoch"]))
            replay_payload = replay_source["payload"]
            replay_epoch = int(replay_payload["epoch"])
            old_protocol = dict(old_payload["protocol"])
            old_protocol.pop("exact_best_min_gain", None)
            old_protocol.pop("significant_min_gain", None)
            old_protocol.update({
                "schedule_id": schedule_id,
                "selection_metric":
                    "validation accuracy (3,000-sample split)",
                "evaluation_role":
                    "model selection and reporting; official test unused",
                "stall_gate": (
                    "100 epochs after raw validation best at epoch >=200"),
                "theta_P_scope": (
                    "raw validation best at or after recipe epoch 200"),
                "validation_best_min_gain": 0.0,
                "post_fork_epochs": 150,
                "optimizer": "SGD(momentum=0.9, weight_decay=0.0005)",
                "model_state_lineage": f"random_init_densenet121_seed_{SEED}",
                "batch_size": 64,
            })
            detector = BestCheckpointStallDetector(
                patience=100, min_gain=0.0, require_arm=True,
                exact_best_patience=True)
            replay_history = list(replay_payload["history"])
            for row in replay_history:
                row_epoch = int(row["epoch"])
                metric = float(row["validation_accuracy"])
                detector.update(row_epoch, metric)
                if row_epoch == 200:
                    detector.arm_stall(row_epoch, metric)
            missing_best_epoch = (
                int(detector.best_epoch)
                if int(detector.best_epoch) != replay_epoch else None)
            migrated_progress = {
                **replay_payload, "protocol": old_protocol,
                "best_stall_detector": detector.state_dict(),
                "replay_missing_best_epoch": missing_best_epoch}
            migrated_best = {
                **replay_payload,
                "kind": "vanilla_validation_best_checkpoint",
                "protocol": old_protocol,
                "replay_missing_best_epoch": missing_best_epoch,
                "placeholder_for_missing_best": missing_best_epoch}
            torch.save(
                migrated_progress, PHASE1_OUTPUT / "checkpoint_latest.pt")
            torch.save(migrated_best, PHASE1_OUTPUT / "checkpoint_best.pt")
            torch.save(
                migrated_best, PHASE1_OUTPUT / "checkpoint_exact_best.pt")
            compatible = [{
                "path": PHASE1_OUTPUT / "checkpoint_latest.pt",
                "payload": migrated_progress,
                "source": f"replay:{replay_source['source']}"}]
            bests = [{
                "path": PHASE1_OUTPUT / "checkpoint_best.pt",
                "payload": migrated_best,
                "source": "migrated-validation-best"}]
            print("Replaying legacy trajectory:", {
                "observed_validation_best_epoch": old_best_epoch,
                "replay_checkpoint_epoch": replay_epoch,
                "epochs_to_observed_best": old_best_epoch - replay_epoch,
            })
        else:
            print("Cannot replay attached legacy run: no compatible full "
                  "checkpoint at or before validation-best epoch",
                  old_best_epoch)
if compatible:
    progress = max(compatible, key=lambda item: int(item["payload"]["epoch"]))
    payload = progress["payload"]
    history = payload["history"]
    post200 = [row for row in history if int(row["epoch"]) >= 200]
    if post200:
        best_row = max(post200, key=lambda row: row["validation_accuracy"])
        best_epoch = int(best_row["epoch"])
    else:
        best_epoch = int(payload["best_stall_detector"]["best_epoch"])
    pending_missing_best = payload.get("replay_missing_best_epoch")
    matching_bests = [item for item in bests
                      if compatible_training_lineage(item["payload"], payload)
                      and int(item["payload"].get("epoch", -1)) == best_epoch]
    if pending_missing_best is not None:
        selected_best = progress
        print("Replaying with historical raw-best metric but missing weights:", {
            "missing_best_epoch": int(pending_missing_best),
            "replay_epoch": int(payload["epoch"]),
            "rule": "must observe a new strict raw validation best before fork",
        })
    elif not matching_bests:
        earlier_bests = [item for item in bests
                         if compatible_training_lineage(
                             item["payload"], payload)
                         and int(item["payload"].get("epoch", -1)) <=
                         best_epoch]
        if earlier_bests:
            # The observed best weights are unavailable, so replay from the
            # closest complete raw-best checkpoint at or before that epoch.
            # Its model/optimizer/scheduler/RNG/loader state remain intact.
            selected_best = max(
                earlier_bests,
                key=lambda item: int(item["payload"]["epoch"]))
            progress = selected_best
            payload = progress["payload"]
            history = payload["history"]
            print("Exact observed validation-best checkpoint is absent; "
                  "replaying from closest compatible raw best:", {
                      "required_epoch": best_epoch,
                      "replay_epoch": int(payload["epoch"]),
                      "source": progress["source"],
                  })
        elif int(payload["epoch"]) == best_epoch:
            selected_best = progress
            print("Repairing best checkpoint from atomic latest checkpoint")
        else:
            raise RuntimeError(
                f"Found resumable epoch {payload['epoch']} but no compatible "
                f"full checkpoint at or before validation-best epoch "
                f"{best_epoch}")
    else:
        selected_best = matching_bests[0]
    copy_checkpoint(progress["path"], PHASE1_OUTPUT / "checkpoint_latest.pt")
    copy_checkpoint(
        selected_best["path"], PHASE1_OUTPUT / "checkpoint_best.pt")
    copy_checkpoint(
        selected_best["path"], PHASE1_OUTPUT / "checkpoint_exact_best.pt")
    resumed_history = payload["history"]
    resumed_post200 = [row for row in resumed_history
                       if int(row["epoch"]) >= 200]
    resumed_best_epoch = (
        int(max(resumed_post200,
                key=lambda row: row["validation_accuracy"])["epoch"])
        if resumed_post200 else 200)
    target_epoch = resumed_best_epoch + 150
    remaining = max(0, target_epoch - int(payload["epoch"]))
    PHASE1_MAX_EPOCH = max(
        PHASE1_MAX_EPOCH, int(payload["epoch"]) + remaining)
    print("Resuming Phase 1:", {
        "checkpoint_epoch": int(payload["epoch"]),
        "validation_best_epoch": resumed_best_epoch,
        "minimum_additional_epochs_if_no_new_best": remaining,
        "source": progress["source"],
    })
else:
    print("No compatible Phase-1 checkpoint in /kaggle/input or the current "
          "working run; starting epoch 0")


In [ ]:
SEED = 1
PHASE1_OUTPUT = RUN_ROOT / "vanilla_stall"
PHASE1_OUTPUT.mkdir(parents=True, exist_ok=True)

phase1_command = [
    sys.executable, "-m", "experiments.run_unified_vanilla_to_stall",
    "--reference-root", str(REFERENCE), "--data-root", str(DATA_ROOT),
    "--output", str(PHASE1_OUTPUT), "--seed", str(SEED),
    "--architecture", "densenet121",
    "--max-epoch", str(PHASE1_MAX_EPOCH), "--batch-size", "64",
    "--validation-samples", "5000", "--trigger-samples", "2000",
    "--tuning-samples", "128", "--lr", "0.1", "--recipe-epochs", "200",
    "--lr-milestones", "100,150", "--lr-gamma", "0.1",
    "--weight-decay", "0.0005", "--stall-patience", "150",
    "--post-fork-epochs", "150"]

phase1_log = (PHASE1_OUTPUT / "notebook_stream.log").open("a", buffering=1)
phase1_env = os.environ.copy()
phase1_env.update(CUDA_VISIBLE_DEVICES="0", PYTHONUNBUFFERED="1",
                  PYTHONPATH=RUNTIME_PYTHONPATH)
phase1_process = subprocess.Popen(
    phase1_command, cwd=REPO, env=phase1_env, stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in phase1_process.stdout:
    phase1_log.write(line); phase1_log.flush()
    print(f"[seed{SEED} vanilla] {line}", end="", flush=True)
phase1_code = phase1_process.wait()
phase1_log.close()
if phase1_code:
    raise RuntimeError(f"unified Vanilla exited with code {phase1_code}")


In [ ]:
phase1_result = json.loads(
    (PHASE1_OUTPUT / "result.json").read_text())
if not phase1_result["plateau_found"]:
    archive = shutil.make_archive(
        str(RUN_ROOT), "gztar", root_dir=RUN_ROOT)
    raise RuntimeError(
        "No confirmed 150-epoch stall by max_epoch. All resumable Phase-1 "
        f"checkpoints were retained in {archive}")

PLATEAU_CHECKPOINT = Path(phase1_result["plateau_checkpoint"])
PLATEAU_PAYLOAD = torch.load(
    PLATEAU_CHECKPOINT, map_location="cpu", weights_only=False)
if PLATEAU_PAYLOAD.get("kind") != "plateau_fork_checkpoint":
    raise RuntimeError("Phase 1 did not produce a plateau fork checkpoint")
PLATEAU_HASH = hashlib.sha256(
    PLATEAU_CHECKPOINT.read_bytes()).hexdigest()
PLATEAU_EPOCH = int(PLATEAU_PAYLOAD["epoch"])
VANILLA_CONTROL = dict(PLATEAU_PAYLOAD["vanilla_control"])
if VANILLA_CONTROL["post_fork_epochs"] != 150:
    raise RuntimeError("Vanilla baseline is not exactly 150 post-fork epochs")
if VANILLA_CONTROL["role"] != "matched_validation_best_to_150_epoch_window":
    raise RuntimeError("Phase-1 Vanilla control has an invalid role")
if int(PLATEAU_PAYLOAD["protocol"]["seed"]) != SEED:
    raise RuntimeError("theta_best seed mismatch")
print("theta_best:", PLATEAU_EPOCH, PLATEAU_HASH)
print(json.dumps(VANILLA_CONTROL, indent=2, sort_keys=True))


## Phase 2 — two projected arms plus matched Vanilla

The Phase-1 trajectory already contains exactly 150 Vanilla epochs after
theta_P. E-driven O and O-only now load the same theta_P hash and each consume
the same 150-SGD-epoch budget. They run concurrently on the two T4 GPUs.


In [ ]:
import threading

OUTPUT = RUN_ROOT / "phase2"
OUTPUT.mkdir(parents=True, exist_ok=True)


In [ ]:
def base_args(output):
    return [
        "--reference-root", str(REFERENCE), "--data-root", str(DATA_ROOT),
        "--plateau-checkpoint", str(PLATEAU_CHECKPOINT),
        "--plateau-checkpoint-hash", PLATEAU_HASH,
        "--output", str(output), "--post-fork-epochs", "150",
        "--seed", "1", "--architecture", "densenet121",
        "--site", "core.features.denseblock3",
        "--batch-size", "64", "--rank", "4",
        "--probe-epsilon", "0.05", "--where-batches", "3",
        "--retrigger-patience", "10",
        "--line-search-scales", "0.0125,0.025,0.05"]

method_names = ("ours_e_driven_o", "o_projection_only")
commands = {
    name: [sys.executable, "-m", "experiments.run_plateau_fork",
           "--method", name] + base_args(OUTPUT / name)
    for name in method_names}

# Resume only arm checkpoints produced from this exact theta_P and protocol.
arm_checkpoints, rejected_arm_checkpoints = discover_checkpoints(
    "/kaggle/input", OUTPUT,
    kind={"plateau_fork_arm_progress", "plateau_fork_arm_best"})
arm_progress = [item for item in arm_checkpoints
                if item["payload"]["kind"] == "plateau_fork_arm_progress"]
arm_bests = [item for item in arm_checkpoints
             if item["payload"]["kind"] == "plateau_fork_arm_best"]
for name in method_names:
    matches = [item for item in [*arm_progress, *arm_bests]
               if item["payload"].get("theta_best_hash") == PLATEAU_HASH
               and item["payload"].get("protocol", {}).get("method") == name
               and item["payload"].get("protocol", {}).get("architecture") ==
                   "CIFAR-DenseNet121"]
    if not matches:
        continue
    progress = max(
        matches, key=lambda item: int(item["payload"]["post_fork_epoch"]))
    best_matches = [item for item in arm_bests
                    if item["payload"].get("theta_best_hash") == PLATEAU_HASH
                    and item["payload"].get("protocol") ==
                    progress["payload"].get("protocol")]
    if not best_matches:
        raise RuntimeError(
            f"Found resumable {name} progress without its best checkpoint")
    best = max(
        best_matches,
        key=lambda item: int(item["payload"]["post_fork_epoch"]))
    arm_output = OUTPUT / name
    arm_output.mkdir(parents=True, exist_ok=True)
    shutil.copy2(progress["path"], arm_output / "checkpoint_latest.pt")
    shutil.copy2(best["path"], arm_output / "checkpoint_best.pt")
    print(f"Resuming {name} at post-fork epoch "
          f"{progress['payload']['post_fork_epoch']}")

def stream(name, process, log):
    for line in process.stdout:
        log.write(line); log.flush()
        print(f"[{name}] {line}", end="", flush=True)

def launch(gpu, name):
    out = OUTPUT / name; out.mkdir(parents=True, exist_ok=True)
    log = (out / "run.log").open("a", buffering=1)
    env = os.environ.copy()
    env.update(CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1",
               PYTHONPATH=RUNTIME_PYTHONPATH)
    process = subprocess.Popen(
        commands[name], cwd=REPO, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1)
    thread = threading.Thread(
        target=stream, args=(name, process, log), daemon=True)
    thread.start()
    print(f"GPU{gpu}: started {name}, pid={process.pid}")
    return name, process, thread, log

def finish(job):
    name, process, thread, log = job
    code = process.wait()
    thread.join(); log.close()
    print(name, "exit=", code)
    return name, code

print("GPU0: E-driven O; GPU1: O-only")
jobs = [launch(0, "ours_e_driven_o"),
        launch(1, "o_projection_only")]
statuses = [finish(job) for job in jobs]
failures = [status for status in statuses if status[1]]
if failures:
    raise RuntimeError(f"failed arms: {failures}")


In [ ]:
results = {
    name: json.loads((OUTPUT / name / "result.json").read_text())
    for name in ("ours_e_driven_o", "o_projection_only")}
for name, result in results.items():
    if result["plateau_checkpoint_hash"] != PLATEAU_HASH:
        raise RuntimeError(f"{name} used another theta_P")
    if result["theta_best_hash"] != PLATEAU_HASH:
        raise RuntimeError(f"{name} used another theta_best")
    if result["post_fork_epochs"] != 150:
        raise RuntimeError(f"{name} did not complete 150 epochs")
    if result.get("protocol", {}).get("architecture") != "CIFAR-DenseNet121":
        raise RuntimeError(f"{name} did not run CIFAR-DenseNet121")
    if not Path(result["best_checkpoint"]).is_file():
        raise RuntimeError(f"{name} did not save checkpoint_best.pt")

VANILLA_CONTROL["plateau_checkpoint_hash"] = PLATEAU_HASH
VANILLA_CONTROL["theta_best_hash"] = PLATEAU_HASH
VANILLA_CONTROL["best_checkpoint"] = str(PLATEAU_CHECKPOINT)
results["vanilla"] = VANILLA_CONTROL
metric_fields = (
    "fork_validation_accuracy", "fork_validation_loss",
    "theta_P_validation_accuracy", "theta_P_validation_loss",
    "final_validation_accuracy", "best_validation_accuracy",
    "final_validation_loss", "best_validation_loss",
    "validation_accuracy_delta", "best_validation_accuracy_delta",
    "epochs_to_best", "training_seconds", "peak_gpu_memory",
    "peak_train_params", "deploy_params", "best_checkpoint",
    "intervention_count", "correction_application_count",
    "correction_application_rate", "intervention_seconds",
    "rollback_count", "retrigger_patience", "interventions")
summary = {
    "architecture": "CIFAR-DenseNet121", "seed": SEED,
    "plateau_epoch": PLATEAU_EPOCH,
    "stall_patience": 150,
    "plateau_checkpoint_hash": PLATEAU_HASH,
    "theta_best_hash": PLATEAU_HASH,
    "post_fork_epochs": 150, "official_test_used": False,
    "phase1_vanilla_control": VANILLA_CONTROL,
    "results": {
        name: {key: result.get(key) for key in metric_fields}
        for name, result in results.items()},
}
(OUTPUT / "summary.json").write_text(
    json.dumps(summary, indent=2, sort_keys=True))
print(json.dumps(summary, indent=2, sort_keys=True))
print("\nmethod                  fork_acc   best_acc  final_acc  epoch_best")
def metric(value):
    return "n/a" if value is None else f"{value:.4f}"

for name in ("vanilla", "ours_e_driven_o", "o_projection_only"):
    row = results[name]
    print(f"{name:23s} {metric(row['fork_validation_accuracy']):>8s} "
          f"{metric(row['best_validation_accuracy']):>9s} "
          f"{metric(row['final_validation_accuracy']):>10s} "
          f"{str(row['epochs_to_best']):>10s}")
archive = shutil.make_archive(str(OUTPUT), "gztar", root_dir=OUTPUT)
print("Archive:", archive)


In [ ]:
full_archive = shutil.make_archive(
    str(RUN_ROOT), "gztar", root_dir=RUN_ROOT)
print("Complete seed archive:", full_archive)
print("Phase-1 latest:", PHASE1_OUTPUT / "checkpoint_latest.pt")
print("Phase-1 best:", PHASE1_OUTPUT / "checkpoint_best.pt")
print("Fork checkpoint:", PLATEAU_CHECKPOINT)
